![Week 5 Day 3](../../assets/banners/w5d3.png)

# Week 5 Day 3 - Solution: A Class Summary Page for Emma

**The exercise:** write `render_class_summary(reports)`, a small self-contained HTML page with the class mean, the grade bands, one bar per question (hardest first) and the flagged answers, all escaped.

Try the exercise yourself first: struggling a little is where the learning happens! Then compare with this one. Everything runs offline with stand-ins (🎭); the 🔑 cells use your real models.

💡 **Fast & cheap run.** Already know the ideas and just want to watch everything run? Put `DATA_SIZE=small` in your `.env` (2 PDFs · Q1–Q3 · 2 students) and keep the default `MODEL_MODE=cheap` (`openai/gpt-6-luna` via OpenRouter, a few cents of API calls). Or set `MODEL_MODE=free` to run on OpenRouter's free models for $0, with the same `OPENROUTER_API_KEY` and no credits: it's rate-limited to 20 requests a minute and 50 a day (1,000 a day once you've ever bought $10 of credits), so pair it with `DATA_SIZE=small`; the model changes from call to call, so results vary; and free providers may log or train on prompts, which is fine for this course's fictional data and never for real students' work. Restart the kernel after editing `.env`; the setup cell below prints what you're running on.

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv
from IPython.display import Markdown, display

from gradewise import datasets
from gradewise.config import settings

load_dotenv(override=True)

# 💡 MODEL_MODE=cheap (default) and MODEL_MODE=free both need just OPENROUTER_API_KEY
for key_name in settings.required_keys:
    api_key = os.getenv(key_name)
    if api_key:
        print(f"✅ {key_name} exists and begins {api_key[:8]}")
    else:
        print(f"❌ {key_name} not set. Most of today's cells still run offline; the few that call a model will")
        print("   show a friendly 'Missing API key' panel. See Part 5 of the setup guide, or setup/SWAP_PROVIDER.md")

# 📦 What this run uses: MODEL_MODE and DATA_SIZE in .env (restart the kernel after changing them)
print(settings.describe_models())
print(datasets.describe())

In [ ]:
import json

from langchain_core.embeddings import DeterministicFakeEmbedding

from gradewise import datasets
from gradewise.config import DATA_DIR, OUTPUT_DIR, settings
from gradewise.console import success
from gradewise.evals.compat import import_pandas
from gradewise.ingestion import ingest
from gradewise.rag import get_retriever
from gradewise.schemas import ExamPaper, ReportCard

pd = import_pandas()  # 💡 = `import pandas as pd`, with a fix for locked-down Windows machines

EXAM_FILE = DATA_DIR / "exams" / "cs_midterm" / "expected_exam.json"
FULL_EXAM = ExamPaper.model_validate_json(EXAM_FILE.read_text(encoding="utf-8"))
QUESTIONS = {q.id: q for q in FULL_EXAM.questions}  # every question, for the one-answer demos
TRUTH = json.loads((DATA_DIR / "answer-sheets" / "synthetic" / "transcripts.json").read_text(encoding="utf-8"))

# 📦 What the batch runs grade: all 15 parts and 10 students, or Q1–Q3 and 2 students with DATA_SIZE=small
exam = datasets.subset_exam(FULL_EXAM)
STUDENTS = datasets.synthetic_students()

# 🆓 A keyword-only (BM25) copy of the CS201 notes: fake embeddings, real keyword search, zero cost
OFFLINE_DB = OUTPUT_DIR / "week5_offline_db"
fake_embeddings = DeterministicFakeEmbedding(size=64)
ingest(
    collection="cs201_offline",
    embeddings=fake_embeddings,
    persist_dir=OFFLINE_DB,
    quiet=True,
    files=datasets.kb_files(),
)  # 📦 all 4 PDFs, or unit1 + the handbook with DATA_SIZE=small
bm25 = get_retriever("cs201_offline", k=4, mode="bm25", embeddings=fake_embeddings, persist_dir=OFFLINE_DB)

success(
    "Ready",
    {
        "Exam": f"{exam.title} ({len(exam.questions)} questions, {exam.total_marks:g} marks)",
        "Students": f"{len(STUDENTS)} synthetic scripts",
        "Offline KB": f"cs201_offline (BM25, {len(datasets.kb_files())} PDFs)",
        "Grading model": settings.grading_model,
    },
)

In [ ]:
from gradewise.grading import build_grading_graph, grade_sheet
from gradewise.grading.fakes import KeywordGraderModel
from gradewise.orchestrator import sheets_from_transcripts

graph = build_grading_graph(KeywordGraderModel(), bm25)  # 🎭 stand-in grader
reports = [grade_sheet(exam, sheet, graph=graph) for sheet in sheets_from_transcripts(exam, students=STUDENTS)]
print(f"{len(reports)} report cards")  # 📦 10, or 2 with DATA_SIZE=small

In [ ]:
import html
from collections import Counter

from IPython.display import HTML

from gradewise.reports.qa_agent import question_statistics


def render_class_summary(reports: list[ReportCard]) -> str:
    """One self-contained HTML page summarising a class. Every string is escaped."""
    stats = question_statistics(reports)
    mean = sum(r.percentage for r in reports) / len(reports)
    bands = Counter(r.grade_band for r in reports)
    band_chips = "".join(f'<span class="chip">{html.escape(b)}: {bands.get(b, 0)}</span>' for b in "ABCDF")
    rows = []
    for qid, row in stats.iterrows():
        q = QUESTIONS.get(qid)
        rows.append(
            f"<tr><td><b>{html.escape(q.number if q else qid)}</b></td>"
            f"<td>{html.escape((q.text if q else '')[:70])}…</td>"
            f"<td>{row['mean']:g} / {row['max_marks']:g}</td>"
            f'<td><div class="bar"><i style="width:{row["mean_pct"]:.0f}%"></i></div></td>'
            f"<td>{int(row['flagged']) or ''}</td></tr>"
        )
    flagged = [
        f"<li>{html.escape(r.student_id)} · {html.escape(g.question_id)}: {html.escape(g.review_reason or '')}</li>"
        for r in reports
        for g in r.results
        if g.needs_review
    ] or ["<li>Nothing is flagged.</li>"]
    return f"""<!doctype html><html><head><meta charset="utf-8"><style>
body {{ font: 14px/1.5 system-ui, sans-serif; color: #1B2140; margin: 0; padding: 20px; background: #EEF0F8; }}
.hero {{ background: #0B1020; color: #E6E9F5; border-radius: 16px; padding: 18px 24px; }}
.hero b {{ font-size: 28px; }}
.chip {{ border: 1px solid #22D3EE; border-radius: 99px; padding: 2px 10px; margin-right: 6px; }}
table {{ width: 100%; border-collapse: collapse; background: #fff; border-radius: 12px; margin-top: 14px; }}
td, th {{ padding: 8px; border-bottom: 1px solid #EEF0F7; text-align: left; }}
.bar {{ height: 8px; background: #EEF0F7; border-radius: 99px; min-width: 120px; overflow: hidden; }}
.bar i {{ display: block; height: 100%; background: linear-gradient(90deg, #7C5CFF, #22D3EE); }}
</style></head><body>
<div class="hero">{html.escape(exam.title)}<br><b>{mean:.1f}%</b> class mean · {len(reports)} students
<br><br>{band_chips}</div>
<table><tr><th>Q</th><th>Question</th><th>Mean</th><th>Mean %</th><th>⚑</th></tr>{"".join(rows)}</table>
<h3>⚑ Awaiting review</h3><ul>{"".join(flagged)}</ul></body></html>"""


page = render_class_summary(reports)
display(HTML(f'<div><iframe srcdoc="{html.escape(page)}" style="width:100%;height:760px;border:0"></iframe></div>'))

Check the escaping: a hostile student ID must come out as text, not markup.

In [ ]:
evil = reports[0].model_copy(update={"student_id": "<script>alert(1)</script>"})
print("escaped:", "<script>alert(1)" not in render_class_summary([evil, *reports[1:]]))

### 🔑 Bonus: follow-up questions with memory
With `checkpointer=InMemorySaver()` and a `thread_id`, the results agent remembers the conversation, so "and which students got it right?" knows what *it* refers to. (We store the reports with the free fake embeddings: the exact tools are what matter here.)

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

from gradewise.errors import GradeWiseError
from gradewise.reports import build_report_agent, rule_based_feedback, store_report

SANDBOX = OUTPUT_DIR / "week5" / "day3_solution"
STORE = {"db": SANDBOX / "gradewise.db", "embeddings": fake_embeddings, "persist_dir": SANDBOX / "vector_db"}
for report in reports:
    store_report(rule_based_feedback(report, exam), exam, **STORE)

try:
    agent = build_report_agent(exam.exam_id, checkpointer=InMemorySaver(), **STORE)
    config = {"configurable": {"thread_id": "emma-1"}}
    for question in ["Which question did the class find hardest?", "And which students got it right?"]:
        state = agent.invoke({"messages": [{"role": "user", "content": question}]}, config)
        display(Markdown(f"**🧑‍🏫 {question}**\n\n{state['messages'][-1].text}"))
except GradeWiseError as err:
    err.show()